### Mosaic Setup

In [0]:
%pip install uv

In [0]:
### https://github.com/astral-sh/uv/issues/8671
# need to use non-default virtual env location for `uv` on Databricks. will piggyback on the existing virtual env Databricks has setup automatically.
# python version specified in pyproject.toml should match Databricks Runtime python version (no known workaround).
import os
path_current = os.getcwd()
os.environ["UV_PROJECT_ENVIRONMENT"] = os.environ["VIRTUAL_ENV"]
path_mosaic = os.path.abspath("/Workspace/Users/karthik.raj@bio-techne.com/mosaic")
os.chdir(path_mosaic)

In [0]:
import os
# Must be set BEFORE `import jax` to take effect
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"  # Use 95% of GPU (default 75%)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # Allocate on-demand, not upfront
# os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # Optional: use CUDA allocator (slower but frees memory)

In [0]:
%%sh
uv add jax[cuda12]
uv sync --group jax-cuda --extra torch-cuda
uv pip install .

### Motif Scaffolding Preparation

1. Creating Target Distogram based on the motif's CB distances
2. Create Motif-Specific Distogram based Loss
3. Create Linker Sampler to sample lengths between motifs

        Final Binder Length = Sum(Linker Lengths) + Sum(Motif Lengths)


In [0]:
import os
# Optional: Set the working directory explicitly to the current workspace folder
import sys
os.chdir("/Workspace/Users/karthik.raj@bio-techne.com/auto_mosaic")

print(f"Current Working Directory: {os.getcwd()}")

In [0]:
import torch

# 1. Force PyTorch to initialize the CUDA driver context immediately
torch.cuda.init() 
print(torch.__version__, torch.version.cuda)
print(torch._C._GLIBCXX_USE_CXX11_ABI)

In [0]:
%%sh
uv pip install flash-attn==2.8.3 --no-build-isolation
uv pip install databricks-sdk  

In [0]:
!python prepare_single.py

## Hyperparameter Grid Search with Replicates (train_ml + MLflow + Optuna)

Sweeps a **small** composite loss-weight grid, and runs **every config at multiple seeds** (`SEEDS`), because a single seed is too noisy to rank configs (the same config was seen to land composite ~1.1 vs a blow-up on init alone). Each Optuna trial = one config; its objective is the **seed-averaged composite** (`mean + RISK_LAMBDA*std`), and the per-seed **std** is logged as the reliability signal. Seeds are shared across configs (common random numbers), so config comparisons stay paired.

Each `(config, seed)` runs `TRAIN_SCRIPT` as a fresh subprocess (clean GPU/JAX state). A seed that hard-crashes is charged `PENALTY_COMPOSITE` so instability counts against the config's mean. **Point `TRAIN_SCRIPT` at the script that has the template-annealing + stabilized free-phase hyperparameters, and run this only after stabilization** — on an unstable base the weight rankings are contaminated.

### MLFlow & Optuna Setup

In [0]:
import mlflow
mlflow.login()
# Create mlflow experiment
experiment_name = f"mosaic_hallu_scaffold_gopher_template_anneal"
full_experiment_name = f"/Users/karthik.raj@bio-techne.com/{experiment_name}"
if not mlflow.get_experiment_by_name(full_experiment_name):
    mlflow.create_experiment(full_experiment_name)
 
# Start mlflow run here to avoid multiple runs being created automatically
mlflow.set_experiment(full_experiment_name)

In [0]:
import shutil
import optuna
def databricks_optuna_storage(
    optuna_file_path: str, tmp_optuna_file_path: str
) -> optuna.storages.JournalStorage:
    """
    Create an Optuna JournalStorage object for Databricks.

    Parameters
    ----------
    optuna_file_path : str
        The path to the Optuna study file.
    tmp_optuna_file_path : str
        The temporary path to store the Optuna study file.

    Returns
    -------
    optuna.storages.JournalStorage
        The JournalStorage object for Databricks.

    Notes
    -----
    - It uses `shutil.copy` to copy the `optuna_file_path` to `tmp_optuna_file_path` if the file exists.
        - This is done because databricks dbfs storage does not allow for append operations
        - https://kb.databricks.com/dbfs/errno95-operation-not-supported
    - The `lock_obj` is created using `optuna.storages.JournalFileOpenLock` to ensure exclusive access to the file.
    - The `file_storage` is created using `optuna.storages.JournalFileStorage` with the `lock_obj`.
    - Finally, the `storage` is created using `optuna.storages.JournalStorage` with the `file_storage`.
    """
    if os.path.isfile(optuna_file_path):
        shutil.copy(optuna_file_path, tmp_optuna_file_path)

    storage = optuna.storages.RDBStorage(
        url=f"sqlite:///{tmp_optuna_file_path}",
        failed_trial_callback=optuna.storages.RetryFailedTrialCallback(max_retry=3),
    )

    return storage

In [0]:
class DatabricksOptunaStorageCallback:
    """
    A callback class for storing Optuna study progress in Databricks.

    Parameters
    ----------
    optuna_file_path : str
        The file path to store the Optuna study progress.
    tmp_optuna_file_path : str
        The temporary file path for storing the Optuna study progress.
    """

    def __init__(self, optuna_file_path: str, tmp_optuna_file_path: str):
        self.optuna_file_path = optuna_file_path
        self.tmp_optuna_file_path = tmp_optuna_file_path

    def __call__(
        self, study: optuna.study.Study, trial: optuna.trial.FrozenTrial
    ) -> None:
        """
        Callback function to store Optuna study progress.

        Parameters
        ----------
        study : optuna.study.Study
            The Optuna study object.
        trial : optuna.trial.FrozenTrial
            The Optuna frozen trial object.

        Returns
        -------
        None
        """
        try:
            shutil.copy(self.tmp_optuna_file_path, self.optuna_file_path)
            print("Optuna save activity succeeded")
        except Exception as e:
            print(f"Optuna save activity failed: {e}")

In [0]:
# Create optuna storage
optuna_file_path = f"/Workspace/Users/karthik.raj@bio-techne.com/auto_mosaic/grid_search/{experiment_name}_hpt.db"
os.makedirs(os.path.dirname(optuna_file_path), exist_ok=True)
tmp_optuna_file_path = f"/tmp/{experiment_name}_hpt.db"

if os.path.exists(tmp_optuna_file_path):
    os.remove(tmp_optuna_file_path)

storage = databricks_optuna_storage(
    optuna_file_path=optuna_file_path, tmp_optuna_file_path=tmp_optuna_file_path
)
databricks_optuna_storage_cb = DatabricksOptunaStorageCallback(
    optuna_file_path=optuna_file_path, tmp_optuna_file_path=tmp_optuna_file_path
)

In [0]:
import json
import math
import os
import subprocess
import tempfile
import statistics

import mlflow
import optuna


# --- Replicate design --------------------------------------------------------
# Each config is run at EVERY seed in SEEDS (paired / common-random-numbers across
# configs) and the value Optuna minimizes is the MEAN composite over seeds. A single
# seed is too noisy to rank configs -- we watched the SAME config land composite ~1.1
# vs a blow-up depending only on the PSSM init -- so we replicate and average, and we
# report the std as the reliability signal.
SEEDS = [21, 42, 43, 44, 77]       # >=3 recommended; add more for tighter estimates
RISK_LAMBDA = 0.5             # objective = mean + RISK_LAMBDA*std ; set >0 (e.g. 0.5) to reward LOW-variance (reliable) configs
PENALTY_COMPOSITE = 4.0       # composite charged to a seed whose run hard-crashes (OOM/nan/exit!=0), so instability counts against the mean

# IMPORTANT: this must be the file that actually contains the template-annealing +
# stabilized (free-phase) hyperparameters. Point it at whichever script is current.
TRAIN_SCRIPT = "train_ml_single_v25.py"

# Baseline weights -- grid points below override only the swept keys; everything else fixed.
BASE_WEIGHTS = {
    'binder_target_contact': 0.5,
    'within_binder_contact': 0.5,
    'inverse_folding_seq_recovery': 6.0,
    'target_binder_pae': 0.05,
    'binder_target_pae': 0.05,
    'within_binder_pae': 0.4,
    'iptm': 0.025,
    'ptm_energy': 0.025,
    'plddt': 0.025,
    'anti_helix': 0.25,
    'bias_sheet': 0.0,            # finding: bias_sheet >= 0.0625 did not help; fixed OFF and dropped from the grid
    'first_motif_distogram': 3.0,
    'first_motif_rmsd': 0.75,
    'specific_contact': 4.0,
    'specific_pae': 0.25,
    'motif_scaffold_contact': 1.0,
    'radius_gyration': 0.75,
    'pose': 0.75,
}

# Reduced grid (findings baked in):
#   motif_scaffold sweet spot is 0.5-1 (weight 2 consistently pulled the motif off the target),
#   pose folds in the 0 ablation vs 0.75 (weight 2 did not separate outcomes),
#   binder_target_contact narrowed to the low end (best runs used 0.5-1).
# 2 x 2 x 2 = 8 configs, each run at len(SEEDS) seeds.
GRID = {
    'motif_scaffold_contact': [0.5, 1],
    'binder_target_contact': [0.5, 1],
    'pose': [0.0, 0.75],
}


def _run_one_seed(weights, seed, tmp_dir, trial_number, parent_run_id, db_host, db_token):
    """Run TRAIN_SCRIPT once at `seed`. Returns (composite | None, log_path, metrics | None).
    composite is None if the subprocess crashed or wrote no result file."""
    result_path = os.path.join(tmp_dir, f"trial_{trial_number}_seed_{seed}_result.json")
    log_path = os.path.join(tmp_dir, f"{experiment_name}_trial_{trial_number}_seed_{seed}.log")
    env = os.environ.copy()
    env["MLFLOW_TRACKING_URI"] = "databricks"
    env["DATABRICKS_HOST"] = db_host
    env["DATABRICKS_TOKEN"] = db_token
    with open(log_path, "a", encoding="utf-8") as log_file:
        log_file.write(f"Trial {trial_number} | seed {seed} | weights: {weights}\n")
        try:
            subprocess.run(
                ["python", TRAIN_SCRIPT, json.dumps(weights),
                 "--seed", str(seed),
                 "--parent_run_id", parent_run_id,
                 "--result_path", result_path],
                check=True, text=True, env=env, stdout=log_file, stderr=subprocess.STDOUT,
            )
        except subprocess.CalledProcessError as e:
            log_file.write(f"\n[seed {seed}] FAILED exit={e.returncode}\n")
            return None, log_path, None
    if not os.path.exists(result_path):
        return None, log_path, None
    with open(result_path) as f:
        metrics = json.load(f)
    return metrics.get("composite_score"), log_path, metrics


with mlflow.start_run(run_name="swp_temp_anneal_reps_val_ESM2") as parent_run:
    parent_run_id = parent_run.info.run_id
    print("Parent Run ID:", parent_run_id)

    # Databricks creds fetched once (not per subprocess).
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    db_host = context.apiUrl().get()
    db_token = context.apiToken().get()

    def objective(trial: optuna.Trial) -> float:
        overrides = {k: trial.suggest_categorical(k, v) for k, v in GRID.items()}
        weights = {**BASE_WEIGHTS, **overrides}
        print(f"Trial {trial.number} config {overrides}  x {len(SEEDS)} seeds")

        composites, per_seed = [], {}
        with tempfile.TemporaryDirectory() as tmp_dir:
            for seed in SEEDS:
                comp, log_path, metrics = _run_one_seed(
                    weights, seed, tmp_dir, trial.number, parent_run_id, db_host, db_token)
                if comp is None:
                    comp = PENALTY_COMPOSITE
                    mlflow.log_artifact(log_path, artifact_path=f"failed/trial_{trial.number}")
                    per_seed[str(seed)] = {"composite_score": comp, "status": "failed"}
                else:
                    mlflow.log_artifact(log_path, artifact_path=f"success/trial_{trial.number}")
                    rec = {"composite_score": comp, "status": "ok"}
                    if metrics:
                        rec.update({k: v for k, v in metrics.items() if k != "composite_score"})
                    per_seed[str(seed)] = rec
                composites.append(comp)
                print(f"    seed {seed}: composite {comp:.3f} ({per_seed[str(seed)]['status']})")

        mean = statistics.fmean(composites)
        std = statistics.pstdev(composites) if len(composites) > 1 else 0.0
        n_fail = sum(1 for r in per_seed.values() if r["status"] == "failed")
        objective_value = mean + RISK_LAMBDA * std

        # per-config aggregate as a nested MLflow run (the raw per-seed train runs still
        # nest under the parent via --parent_run_id inside TRAIN_SCRIPT).
        with mlflow.start_run(run_name=f"config_{trial.number}", nested=True):
            mlflow.log_params({**overrides, "seeds": ",".join(map(str, SEEDS))})
            mlflow.log_metrics({
                "composite_mean": mean, "composite_std": std,
                "composite_min": min(composites), "composite_max": max(composites),
                "n_failed": n_fail, "objective": objective_value,
            })

        trial.set_user_attr("per_seed", per_seed)
        trial.set_user_attr("composite_mean", mean)
        trial.set_user_attr("composite_std", std)
        trial.set_user_attr("n_failed", n_fail)
        print(f"  -> mean {mean:.3f}  std {std:.3f}  min {min(composites):.3f}  max {max(composites):.3f}  fails {n_fail}/{len(SEEDS)}")
        return objective_value

    try:
        optuna.delete_study(study_name=f"{experiment_name}_hpt", storage=storage)
    except KeyError:
        pass  # study doesn't exist yet -- nothing to delete

    study = optuna.create_study(
        study_name=f"{experiment_name}_hpt",
        storage=storage,
        load_if_exists=True,
        direction="minimize",
        sampler=optuna.samplers.GridSampler(search_space=GRID),
    )
    # GridSampler enumerates CONFIGS (seeds are handled inside objective).
    n_trials = math.prod(len(v) for v in GRID.values())
    print(f"Configs: {n_trials}  x  seeds: {len(SEEDS)}  =  {n_trials * len(SEEDS)} total runs")
    study.optimize(objective, n_trials=n_trials, callbacks=[databricks_optuna_storage_cb])

# ---- ranked summary: configs by mean composite, with std as the reliability column ----
print("\n=== configs ranked by mean composite (lower = better; std = reliability) ===")
for t in sorted(study.trials, key=lambda t: t.user_attrs.get("composite_mean", float("inf"))):
    ua = t.user_attrs
    print(f"{t.params}  mean={ua.get('composite_mean', float('nan')):.3f}  "
          f"std={ua.get('composite_std', float('nan')):.3f}  fails={ua.get('n_failed')}/{len(SEEDS)}")
print(f"\nBest by objective (mean + {RISK_LAMBDA}*std): {study.best_params}  ({study.best_value:.4f})")
